# Tema 2 · Clusterización — Segmentación de Clientes
## Ejercicio del sábado (plantilla) · Dataset: **Credit Card Dataset for Clustering**

**Caso de negocio.** Un emisor de tarjetas de crédito quiere segmentar a sus
~9.000 tarjetahabientes según su **comportamiento financiero** (balance, compras,
adelantos de efectivo, pagos, límite de crédito, etc.) para diseñar estrategias
comerciales diferenciadas. No hay etiqueta objetivo: aplicarás **los 3
algoritmos vistos en la demo** (K-Means, jerárquico y DBSCAN) para descubrir
segmentos, los compararás con métricas de calidad, y te quedarás con uno para
el perfilamiento de negocio.

> **Cómo se evalúa.** Completa las celdas `# TODO:` siguiendo las instrucciones.
> Las **secciones 1 y 2 ya están resueltas** como ejemplo. Al final respondes
> preguntas de negocio — **no basta con "dio mejor la métrica"**: se espera que
> justifiques cada respuesta conectándola con el caso de negocio.

Estructura (misma lógica que la demo del viernes):
1. Carga y exploración ✅ *(resuelto)*
2. Limpieza y selección de variables + escalado ✅ *(resuelto)*
3. Elección de K para K-Means (codo + silhouette) — **TODO**
4. Entrenamiento de K-Means — **TODO**
5. Entrenamiento de Clustering Jerárquico — **TODO**
6. Entrenamiento de DBSCAN — **TODO**
7. Evaluación de calidad: comparar los 3 con silhouette, Davies-Bouldin y Calinski-Harabasz — **TODO**
8. Visualización de clusters de los 3 (PCA a 2D) — **TODO**
9. Perfilamiento — descriptiva de los grupos (**obligatoria**, sin importar el algoritmo elegido) — **TODO**
10. Conclusión de negocio — **preguntas de negocio, con justificación**

## 0. Semilla personal (anti-copia)

In [1]:
cedula = 1020304050  # <-- REEMPLAZA por tu número de cédula completo
import numpy as np
np.random.seed(cedula)
print(f"Semilla fijada en {cedula}")

Semilla fijada en 1020304050


## Diccionario de datos — `CC_GENERAL.csv`

| Variable | Descripción | Tipo | Rol |
|---|---|---|---|
| `CUST_ID` | Identificador único del cliente | Texto | Se descarta |
| `BALANCE` | Saldo disponible en la cuenta para hacer compras | Numérica | Feature |
| `BALANCE_FREQUENCY` | Qué tan frecuentemente se actualiza el balance (0 = poco, 1 = muy frecuente) | Numérica (0–1) | Feature |
| `PURCHASES` | Monto total de compras realizadas | Numérica | Feature |
| `ONEOFF_PURCHASES` | Monto de compras hechas de una sola vez (no a cuotas) | Numérica | Feature |
| `INSTALLMENTS_PURCHASES` | Monto de compras hechas a cuotas | Numérica | Feature |
| `CASH_ADVANCE` | Monto de adelantos de efectivo tomados con la tarjeta | Numérica | Feature |
| `PURCHASES_FREQUENCY` | Qué tan frecuentemente se hacen compras (0–1) | Numérica | Feature |
| `ONEOFF_PURCHASES_FREQUENCY` | Frecuencia de compras de una sola vez (0–1) | Numérica | Feature |
| `PURCHASES_INSTALLMENTS_FREQUENCY` | Frecuencia de compras a cuotas (0–1) | Numérica | Feature |
| `CASH_ADVANCE_FREQUENCY` | Frecuencia de adelantos de efectivo (0–1) | Numérica | Feature |
| `CASH_ADVANCE_TRX` | Número de transacciones de adelanto de efectivo | Numérica | Feature |
| `PURCHASES_TRX` | Número de transacciones de compra | Numérica | Feature |
| `CREDIT_LIMIT` | Límite de crédito de la tarjeta | Numérica | Feature (tiene nulos) |
| `PAYMENTS` | Monto total de pagos realizados por el cliente | Numérica | Feature |
| `MINIMUM_PAYMENTS` | Monto mínimo de pagos realizados | Numérica | Feature (tiene nulos) |
| `PRC_FULL_PAYMENT` | Porcentaje de veces que el cliente pagó el total de la deuda | Numérica (0–1) | Feature |
| `TENURE` | Antigüedad del cliente con la tarjeta, en meses | Numérica | Feature |

**No hay target** — es aprendizaje no supervisado. Todas las variables están en
escalas muy distintas (montos vs. frecuencias 0–1 vs. conteos) → **escalado
obligatorio** antes de clusterizar. `CREDIT_LIMIT` y `MINIMUM_PAYMENTS` tienen
nulos que se imputan en la sección 2.

## 1. Carga y exploración  ✅ *(resuelto)*

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../../data/02-clusterizacion/CC_GENERAL.csv")
print("Dimensiones:", df.shape)
df.head()

Dimensiones: (8950, 18)


,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [3]:
df = df.sample(frac=0.85, random_state=cedula).reset_index(drop=True)
print("Dimensiones tras el muestreo:", df.shape)
df.info()

Dimensiones tras el muestreo: (7608, 18)
<class 'pandas.DataFrame'>
RangeIndex: 7608 entries, 0 to 7607
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CUST_ID                           7608 non-null   str    
 1   BALANCE                           7608 non-null   float64
 2   BALANCE_FREQUENCY                 7608 non-null   float64
 3   PURCHASES                         7608 non-null   float64
 4   ONEOFF_PURCHASES                  7608 non-null   float64
 5   INSTALLMENTS_PURCHASES            7608 non-null   float64
 6   CASH_ADVANCE                      7608 non-null   float64
 7   PURCHASES_FREQUENCY               7608 non-null   float64
 8   ONEOFF_PURCHASES_FREQUENCY        7608 non-null   float64
 9   PURCHASES_INSTALLMENTS_FREQUENCY  7608 non-null   float64
 10  CASH_ADVANCE_FREQUENCY            7608 non-null   float64
 11  CASH_ADVANCE_TRX                  7608 

## 2. Limpieza y selección de variables + escalado  ✅ *(resuelto)*

Este dataset tiene **nulos** (en `CREDIT_LIMIT` y `MINIMUM_PAYMENTS`) y variables
en **escalas muy dispares** (balances de miles frente a frecuencias entre 0 y 1).
Dejamos resuelta la limpieza y el escalado; tú los usarás a partir de la sección 3.

### 2.1 Quitar el identificador y tratar nulos
`CUST_ID` es un identificador (no aporta al clustering). Imputamos los nulos con
la **mediana** de cada columna.

In [4]:
df = df.drop(columns=["CUST_ID"])
print("Nulos antes:", df.isna().sum().sum())
df = df.fillna(df.median(numeric_only=True))
print("Nulos después:", df.isna().sum().sum())

Nulos antes: 280
Nulos después: 0


### 2.2 Escalado
Los algoritmos de clustering usan distancias euclídeas, así que
**estandarizamos todas las variables** con `StandardScaler` (si no, `BALANCE`
dominaría sobre las frecuencias).

In [5]:
from sklearn.preprocessing import StandardScaler

X = df.copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Shape escalado:", X_scaled.shape)
print("Media≈0:", np.round(X_scaled.mean(), 3), "| std≈1:", np.round(X_scaled.std(), 3))

Shape escalado: (7608, 17)
Media≈0: 0.0 | std≈1: 1.0


---
# A partir de aquí, completa tú el análisis (secciones 3–10)
---

## 3. Elección de K para K-Means  — **TODO**

Determina un buen número de clusters. Recorre varios valores de K (p. ej. 2 a 10)
y calcula, para cada uno:
- la **inercia** (`KMeans(...).inertia_`) → gráfico del **método del codo**;
- el **silhouette score** (`silhouette_score(X_scaled, labels)`).

Grafica ambas curvas y decide tu K. Usa `random_state=cedula` y `n_init=10`.
Vas a reutilizar este mismo K para el jerárquico en la sección 5.

In [6]:
# TODO: from sklearn.cluster import KMeans
#       from sklearn.metrics import silhouette_score
# Recorre K en range(2, 11), guarda inercia y silhouette, y grafica ambas curvas.

## 4. Entrenamiento de K-Means  — **TODO**

Entrena K-Means con el K que elegiste (`random_state=cedula`, `n_init=10`) y
guarda las etiquetas en una nueva columna `df["Cluster_KMeans"]`. Muestra
cuántos clientes cayeron en cada cluster.

In [7]:
# TODO: entrena KMeans con tu K y asigna df["Cluster_KMeans"]

## 5. Entrenamiento de Clustering Jerárquico  — **TODO**

Entrena `AgglomerativeClustering` sobre el mismo `X_scaled`, con
`linkage="ward"` y el **mismo K** que elegiste para K-Means (para poder
comparar en igualdad de condiciones). Guarda las etiquetas en
`df["Cluster_Jerarquico"]`.

> Opcional: antes de decidir, grafica el dendrograma
> (`scipy.cluster.hierarchy.dendrogram`) como en la demo.

In [8]:
# TODO: from sklearn.cluster import AgglomerativeClustering
# Entrena con linkage="ward" y tu K, asigna df["Cluster_Jerarquico"]

## 6. Entrenamiento de DBSCAN  — **TODO**

DBSCAN no recibe K. Elige `eps` y `min_samples` con el gráfico de
**k-distancia** (`NearestNeighbors`), igual que en la demo: para cada punto,
la distancia a su `min_samples`-ésimo vecino más cercano, ordenada de menor a
mayor — el "codo" de esa curva es tu punto de partida para `eps`. Guarda las
etiquetas en `df["Cluster_DBSCAN"]`.

> Cuidado: si casi todos los puntos quedan como ruido (`-1`) o todos caen en
> un solo cluster, ajusta `eps` y vuelve a intentar. Con 17 variables, el
> `eps` que funcionó en la demo (2 variables) **no** te va a servir igual —
> tienes que recalcular tu propio gráfico de k-distancia sobre estos datos.

In [9]:
# TODO: from sklearn.cluster import DBSCAN
#       from sklearn.neighbors import NearestNeighbors
# Grafica la curva de k-distancia, elige eps y min_samples,
# entrena DBSCAN y asigna df["Cluster_DBSCAN"]

## 7. Evaluación de calidad — comparar los 3 algoritmos  — **TODO**

Calcula, para **K-Means, Jerárquico y DBSCAN**, las tres métricas internas
que aplican a cualquier algoritmo: **silhouette**, **Davies-Bouldin** y
**Calinski-Harabasz** (`sklearn.metrics`). Para DBSCAN, recuerda **excluir
los puntos de ruido** (`label != -1`) antes de calcular las métricas — igual
que en la demo. Arma una tabla comparativa (un `DataFrame` con una fila por
algoritmo, como en la demo) con las 3 métricas y el número de clusters
encontrado por cada uno.

In [10]:
# TODO: from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
# Calcula las 3 métricas para Cluster_KMeans, Cluster_Jerarquico y Cluster_DBSCAN,
# arma una tabla comparativa (una fila por algoritmo).

## 8. Visualización de clusters  — **TODO**

Como hay 17 variables, no se pueden graficar directamente en 2D. Usa **PCA**
(`sklearn.decomposition.PCA`) para proyectar a 2 componentes — pero antes de
confiar en ese gráfico, **valida cuánta información real representa**.

### 8.1 ¿Cuánta varianza explican 2 componentes?  — obligatorio antes de graficar

Ajusta un PCA con **todos los componentes posibles** (no solo 2) sobre
`X_scaled`, y grafica la varianza explicada por componente y la **varianza
acumulada** (*scree plot*). Reporta explícitamente qué porcentaje de la
varianza total capturan **solo los primeros 2 componentes**
(`pca.explained_variance_ratio_[:2].sum()`).

> Con 17 variables de comportamiento financiero es muy probable que 2
> componentes capturen **menos del 50-60%** de la varianza. Eso no invalida
> el ejercicio, pero sí significa que el scatter 2D es una **simplificación**
> — dos puntos que se ven cerca en el gráfico podrían estar lejos en las
> otras 15 dimensiones que no se están mostrando.

### 8.2 Scatter de los 3 algoritmos

Con las coordenadas de PCA a 2D, grafica los **3 algoritmos lado a lado**
(3 subplots en una fila), coloreando los puntos por cada columna de cluster
— igual que en la demo. **En el título o pie del gráfico, incluye el %
de varianza explicada** que calculaste en 8.1, para que quien lo lea sepa
qué tan confiable es leer distancias directamente del scatter.

> Pista: `PCA(n_components=2).fit_transform(X_scaled)` te da las coordenadas
> 2D para el scatter; el `PCA` sin `n_components` fijo (o con
> `n_components=X_scaled.shape[1]`) es el que necesitas para el scree plot.

In [11]:
# TODO 8.1: ajusta PCA con todos los componentes, grafica varianza explicada
#           (por componente y acumulada), y calcula el % que capturan los
#           primeros 2 componentes.

In [12]:
# TODO 8.2: proyecta a 2D con PCA(n_components=2) y grafica los clusters de
#           los 3 algoritmos, uno junto al otro. Incluye el % de varianza
#           explicada calculado en 8.1 en el título o como texto en la figura.

**Si el % de varianza de 8.1 fue bajo (p. ej. menos del 50%)**, no tomes la
separación visual del scatter como el criterio principal para elegir el
mejor algoritmo en la sección 7 — apóyate más en las 3 métricas de calidad
(que se calculan sobre las 17 variables originales, no sobre la proyección
2D) y usa el scatter solo como apoyo visual, aclarándolo así en tu
justificación de la sección 10.

## 9. Perfilamiento — descriptiva de los grupos  — **TODO (obligatorio)**

Esta sección **se hace sí o sí, sin importar qué algoritmo hayas elegido**
como mejor en la sección 7: la descriptiva de los grupos es el paso que le da
sentido de negocio a todo lo anterior — sin ella, el clustering es solo
números.

### 9.1 Descriptiva de todos los clusters

Con el algoritmo que decidiste como mejor (apóyate en las métricas **y** en
la visualización, no solo en un número), calcula la **media de cada
variable por cluster** (`df.groupby(...).mean()`) para caracterizar los
segmentos. Identifica qué distingue a cada grupo (¿alto balance?, ¿muchas
compras?, ¿usan adelantos de efectivo?, ¿pagan completo?).

In [13]:
# TODO: tabla de medias por cluster del algoritmo elegido (obligatorio), e interpretación

### 9.2 Atípicos — sacarlos, mirarlos y describirlos  — **TODO (obligatorio)**

No basta con decir "hay atípicos" — hay que **sacarlos de la base y mirarlos
uno por uno**, no solo su promedio.

- Si elegiste **DBSCAN**: filtra las filas con `Cluster_DBSCAN == -1`
  (`df[df["Cluster_DBSCAN"] == -1]`).
- Si elegiste **K-Means o Jerárquico** (no tienen una etiqueta de "ruido"
  explícita): identifica el **cluster más pequeño o más extremo** —el que
  quede más lejos del resto en el PCA de la sección 8, o el de menor
  tamaño— y trátalo como tu grupo atípico a inspeccionar.

Para ese grupo:
1. Muéstralo completo (`.head(15)` o similar, con **todas** las columnas
   originales, no solo el promedio) y compáralo con `df.describe()` general
   para ver en qué variables se sale del rango típico.
2. Para **cada patrón atípico** que encuentres, decide: **¿es un cliente
   real e inusual** (p. ej. gasta muchísimo, o casi no usa la tarjeta), **o
   parece un error de datos** (valores imposibles, todo en cero, cifras que
   no cuadran entre sí)? Escríbelo en una celda Markdown, con el patrón
   concreto que encontraste como evidencia — no una respuesta genérica.

In [14]:
# TODO: extrae y muestra las filas atípicas (ruido de DBSCAN, o el cluster
#       más pequeño/extremo de tu algoritmo elegido) con TODAS sus columnas,
#       y compáralas contra df.describe() general.

_(tu descripción de los patrones atípicos encontrados, y si te parecen
clientes reales o errores de datos, va aquí)_

## 10. Conclusión de negocio  — **responde en texto, con justificación**

Estas preguntas son de **negocio**, no de código: imagina que se las
respondes al área comercial del emisor de tarjetas, que no sabe qué es
silhouette ni DBSCAN. 3–5 frases por pregunta.

1. **Bautiza cada cluster con un nombre de negocio** (p. ej. "clientes
   premium de bajo riesgo", "usuarios de cash advance", "compradores a
   cuotas"…). Este es el resultado central del ejercicio: si alguien del
   área comercial solo lee esta respuesta, debe entender **quiénes son** tus
   segmentos sin mirar una sola métrica.

2. **¿Cuál segmento es más valioso para el negocio (mayor ingreso
   potencial, mejor comportamiento de pago), y cuál representa mayor riesgo
   o costo?** Justifica con lo que viste en la descriptiva de la sección 9.

3. **¿Qué acción comercial distinta recomendarías para cada segmento?**
   (p. ej. aumento de cupo, producto de fidelización, alerta de riesgo,
   oferta de educación financiera). No repitas la misma acción para dos
   segmentos distintos.

4. **Si el emisor de la tarjeta solo tuviera presupuesto para una campaña
   este trimestre, ¿en qué segmento la invertirías primero, y qué resultado
   esperarías?**

5. **Con base en lo que encontraste en la sección 9.2, ¿qué harías con esos
   clientes atípicos desde el negocio?** Si concluiste que son clientes
   reales e inusuales (p. ej. muy alto valor), la respuesta es distinta
   (oferta VIP, cuenta ejecutiva dedicada) que si concluiste que son errores
   de datos (excluir de la campaña hasta validar el dato, marcar para
   revisión del equipo de calidad de datos).

**Nota técnica breve** (2–3 frases cada una, esto sí puede ser más técnico):

6. **¿Con cuál de los 3 algoritmos te quedaste, y por qué?** — una frase con
   la métrica que más pesó en tu decisión y una frase con qué viste en la
   visualización.
7. **¿Por qué fue imprescindible escalar las variables antes de aplicar
   estos algoritmos en este dataset en particular?**